In [99]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier
import pickle
import warnings

warnings.filterwarnings('ignore')

In [100]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [101]:
df = df.drop(["customerID"], axis=1)

In [102]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors='coerce')

In [103]:
df.dropna(subset=["TotalCharges"], inplace=True)
df.drop(df[df["tenure"] == 0].index, axis=0, inplace=True, errors='ignore')

In [104]:
#df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1:"Yes"})

In [105]:
X = df.drop(columns=["Churn"])
y = df["Churn"].map({"No": 0, "Yes": 1})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [106]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [col for col in X_train.columns if col not in num_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols)    
    ]
)

In [107]:
model = LGBMClassifier(
    random_state=42, 
    verbose=-1,
    subsample=0.8,
    num_leaves=15,
    n_estimators=300,
    min_child_samples=30,
    max_depth=3,
    learning_rate=0.01,
    colsample_bytree=1.0
)

In [108]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", model)
])

pipeline.fit(X_train, y_train)

y_probs = pipeline.predict_proba(X_test)[:, 1]
threshold_value = 0.32
y_pred_custom = (y_probs >= threshold_value).astype(int)

In [109]:
with open("model.pkl", "wb") as file:
    pickle.dump(pipeline, file)